# Informe completo: comparación señales sintéticas vs reales
Este notebook calcula métricas, genera figuras y guarda resultados para las comparaciones Real vs Sintético (AF y NSR).


In [14]:
# Mostrar ambas tablas generadas (P09 processed vs Article y P09/P10 vs Article)


In [15]:
# --- Imports consolidados ---
import os, pandas as pd
import os
import json
import math
import warnings
import logging
import numpy as np
import pandas as pd
from pathlib import Path
FS = 300.0  # sampling frequency (Hz) - cambiar si tus datos tienen otro fs
MAX_SAMPLE = 1024  # máximo número de señales a usar por métrica para velocidad
N_BOOT = 100  # bootstrap reps (rápido)
N_PERM = 100  # permutaciones (rápido)
SEED = 42
PSD_MAX_FREQ = min(200.0, FS/2.0) 
# Scipy
from scipy.spatial.distance import cdist
from scipy.signal import welch, find_peaks, correlate
from scipy.stats import ks_2samp, wasserstein_distance, skew, kurtosis
from scipy.linalg import sqrtm

# Matplotlib para plots headless
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# sklearn
from sklearn.decomposition import PCA
from sklearn.decomposition import PCA as SKPCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score, silhouette_score, accuracy_score

# Intenta importar UMAP (opcional)
try:
    import umap as _umap
    UMAP_AVAILABLE = True
except Exception:
    _umap = None
    UMAP_AVAILABLE = False

# Configuración y utilidades globales
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
np.seterr(all='ignore')

# Aleatoriedad reproducible
rng = np.random.default_rng(SEED)

# Directorios de salida específicos para p09
OUT_DIR = 'compare_out_p09'
AF_DIR = os.path.join(OUT_DIR, 'pretty_AF')
NSR_DIR = os.path.join(OUT_DIR, 'pretty_NSR')

# Crear directorios de salida
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(AF_DIR, exist_ok=True)
os.makedirs(NSR_DIR, exist_ok=True)


In [16]:
# Rutas de archivos (reales y sintéticos) — carga segura para ejecución headless
paths = {
  'AF_real': 'PULSOVITAL/Metricas/sssd_article_AF_proc_proc_3000.npy',
  'AF_synth': 'PULSOVITAL/Metricas/1024seq_AF.npy',
  'NSR_real': 'PULSOVITAL/Metricas/sssd_article_NSR_proc_proc_3000.npy',
  'NSR_synth': 'PULSOVITAL/Metricas/1024seq_NSR.npy'
}
for k,p in paths.items():
    print(k, os.path.exists(p), p)

# cargar arrays de forma segura: si falta un archivo, crear marcador vacío para no interrumpir la ejecución
def try_load(p):
    if os.path.exists(p):
        try:
            return np.load(p)
        except Exception as e:
            print('Error loading', p, e)
            return np.zeros((0, 3000))
    else:
        print('Warning: missing', p)
        return np.zeros((0, 3000))

AFR = try_load(paths['AF_real'])
AFS = try_load(paths['AF_synth'])
NSRR = try_load(paths['NSR_real'])
NSRS = try_load(paths['NSR_synth'])
print('shapes:', AFR.shape, AFS.shape, NSRR.shape, NSRS.shape)

AF_real False PULSOVITAL/Metricas/sssd_article_AF_proc_proc_3000.npy
AF_synth False PULSOVITAL/Metricas/1024seq_AF.npy
NSR_real False PULSOVITAL/Metricas/sssd_article_NSR_proc_proc_3000.npy
NSR_synth False PULSOVITAL/Metricas/1024seq_NSR.npy
shapes: (0, 3000) (0, 3000) (0, 3000) (0, 3000)


In [17]:
# Normalize shapes

def prepare(A):
    A = np.asarray(A)
    if A.ndim==3:
        A = A.reshape(A.shape[0], -1)
    elif A.ndim==1:
        A = A.reshape(1, -1)
    elif A.ndim==2 and A.shape[1]==1:
        A = A.reshape(A.shape[0], -1)
    # z-normalize per sample; avoid div by zero
    A = (A - A.mean(axis=1, keepdims=True)) / (A.std(axis=1, keepdims=True) + 1e-8)
    return A

AFR_p = prepare(AFR)
AFS_p = prepare(AFS)
NSRR_p = prepare(NSRR)
NSRS_p = prepare(NSRS)

# sanitize NaNs/Infs to keep sklearn happy
AFR_p = np.nan_to_num(AFR_p, nan=0.0, posinf=0.0, neginf=0.0)
AFS_p = np.nan_to_num(AFS_p, nan=0.0, posinf=0.0, neginf=0.0)
NSRR_p = np.nan_to_num(NSRR_p, nan=0.0, posinf=0.0, neginf=0.0)
NSRS_p = np.nan_to_num(NSRS_p, nan=0.0, posinf=0.0, neginf=0.0)

print('prepared shapes:', AFR_p.shape, AFS_p.shape)

prepared shapes: (0, 3000) (0, 3000)


In [18]:
# --- Métricas consolidadas ---
# Funciones Rápidas
def median_rbf_sigma(X, Y, max_pairs=10000):
    X = np.asarray(X); Y = np.asarray(Y)
    # handle empty inputs safely
    if X.size == 0 or Y.size == 0:
        return 1.0
    # ensure 2D shape: (n_samples, n_features)
    try:
        X = X.reshape(len(X), -1)
        Y = Y.reshape(len(Y), -1)
    except Exception:
        X = np.atleast_2d(X)
        Y = np.atleast_2d(Y)
    n1 = min(len(X), MAX_SAMPLE)
    n2 = min(len(Y), MAX_SAMPLE)
    Xs = X[:n1]
    Ys = Y[:n2]
    Z = np.vstack([Xs, Ys])
    m = Z.shape[0]
    if m <= 1:
        return 1.0
    inds = rng.choice(m, size=min(m, 2000), replace=False)
    sub = Z[inds]
    d = cdist(sub, sub, 'euclidean')
    tri = np.triu_indices_from(d,1)
    if len(tri[0]) == 0:
        return 1.0
    med = np.median(d[tri])
    return float(med) if med>0 else 1.0


def rbf_mmd2(X, Y, sigma):
    X = np.asarray(X)
    Y = np.asarray(Y)
    if X.size == 0 or Y.size == 0:
        return float('nan')
    try:
        X = X.reshape(len(X), -1)
        Y = Y.reshape(len(Y), -1)
    except Exception:
        X = np.atleast_2d(X)
        Y = np.atleast_2d(Y)
    Kxx = np.exp(-cdist(X,X,'sqeuclidean')/(2*sigma**2))
    Kyy = np.exp(-cdist(Y,Y,'sqeuclidean')/(2*sigma**2))
    Kxy = np.exp(-cdist(X,Y,'sqeuclidean')/(2*sigma**2))
    m = X.shape[0]; n = Y.shape[0]
    sum_x = (np.sum(Kxx) - np.sum(np.diag(Kxx))) / (m*(m-1)) if m>1 else 0.0
    sum_y = (np.sum(Kyy) - np.sum(np.diag(Kyy))) / (n*(n-1)) if n>1 else 0.0
    sum_xy = np.sum(Kxy) / (m*n) if m>0 and n>0 else 0.0
    return float(sum_x + sum_y - 2*sum_xy)


def energy_distance(X, Y):
    X = np.asarray(X); Y = np.asarray(Y)
    if X.size == 0 or Y.size == 0:
        return float('nan')
    try:
        X = X.reshape(len(X), -1); Y = Y.reshape(len(Y), -1)
    except Exception:
        X = np.atleast_2d(X); Y = np.atleast_2d(Y)
    dx = cdist(X, X); dy = cdist(Y, Y); dxy = cdist(X, Y)
    m = X.shape[0]; n = Y.shape[0]
    ex = np.sum(dx)/(m*m) if m>0 else 0.0; ey = np.sum(dy)/(n*n) if n>0 else 0.0; exy = np.sum(dxy)/(m*n) if m>0 and n>0 else 0.0
    return float(2*exy - ex - ey)


def first_pc_proj(X):
    X = np.asarray(X)
    if X.size == 0:
        return np.array([])
    try:
        X = X.reshape(len(X), -1)
    except Exception:
        X = np.atleast_2d(X)
    Xc = X - X.mean(axis=1, keepdims=True)
    try:
        u,s,vt = np.linalg.svd(Xc, full_matrices=False)
        pc = vt[0]
        return Xc.dot(pc)
    except Exception:
        return X.mean(axis=1)

# Pairwise summary
def compute_pair_metrics(R, S, name, outdir):
    os.makedirs(outdir, exist_ok=True)
    Rsub = R[:MAX_SAMPLE] if len(R)>MAX_SAMPLE else R
    Ssub = S[:MAX_SAMPLE] if len(S)>MAX_SAMPLE else S
    sigma = median_rbf_sigma(Rsub, Ssub)
    mmd2 = rbf_mmd2(Rsub, Ssub, sigma)
    ed = energy_distance(Rsub, Ssub)
    # projection stats
    rproj = first_pc_proj(Rsub)
    sproj = first_pc_proj(Ssub)
    ks_stat, ks_p = ks_2samp(rproj, sproj)
    w = wasserstein_distance(rproj, sproj)
    mean_diff = float(np.mean(rproj)-np.mean(sproj))
    # PSD distance
    def mean_psd(A):
        ps = []
        for s in A[:min(len(A),MAX_SAMPLE)]:
            f,p = welch(s, fs=FS, nperseg=1024)
            ps.append(p)
        return f, np.mean(ps, axis=0)
    f_r, p_r = mean_psd(Rsub)
    f_s, p_s = mean_psd(Ssub)
    psd_l2 = float(np.linalg.norm(p_r - p_s))
    # discriminator AUC
    try:
        X = np.vstack([Rsub[:MAX_SAMPLE], Ssub[:MAX_SAMPLE]])
        y = np.array([0]*min(len(Rsub),MAX_SAMPLE) + [1]*min(len(Ssub),MAX_SAMPLE))
        pca = PCA(n_components=min(50, X.shape[1]))
        Xr = pca.fit_transform(X)
        auc = float(np.mean(cross_val_score(LogisticRegression(max_iter=200), Xr, y, cv=5, scoring='roc_auc')) )
    except Exception:
        auc = float('nan')
    res = dict(name=name, mmd2=mmd2, mmd_sigma=sigma, energy=ed, ks_stat=float(ks_stat), ks_p=float(ks_p), wasserstein=float(w), mean_diff_proj=mean_diff, psd_l2=psd_l2, discriminator_auc=auc)
    outcsv = os.path.join(outdir, f'{name}_summary.csv')
    with open(outcsv,'w',encoding='utf-8') as fh:
        fh.write('metric,value\n')
        for k,v in res.items():
            fh.write(f'{k},{v}\n')
    return res

# Safety wrapper to prevent accidental real-vs-real comparisons
def safe_compute_pair_metrics(R, R_label, S, S_label, name, outdir):
    try:
        if isinstance(R_label, str) and isinstance(S_label, str):
            if R_label.lower().startswith('real') and S_label.lower().startswith('real'):
                raise ValueError(f"Refusing to compare two REAL datasets: {R_label} vs {S_label}")
    except Exception:
        # if label checking fails for any reason, fall back to safe behavior and refuse when both arrays look like real (heuristic)
        pass
    return compute_pair_metrics(R, S, name, outdir)

# additional global guard: replace compute_pair_metrics symbol with a guarded wrapper to prevent accidental calls later in the notebook

def _is_known_real_array(A):
    try:
        if 'AFR_p' in globals() and 'NSRR_p' in globals():
            return np.array_equal(A, AFR_p) or np.array_equal(A, NSRR_p)
        return False
    except Exception:
        return False

_compute_pair_metrics_orig = compute_pair_metrics

def _guarded_compute_pair_metrics(R, S, name, outdir):
    try:
        if _is_known_real_array(R) and _is_known_real_array(S):
            raise ValueError(f"Refusing to compare two REAL datasets (guard): {name}")
    except Exception:
        # if checking fails, be conservative and refuse if both are numpy arrays and have similar shapes
        try:
            if isinstance(R, np.ndarray) and isinstance(S, np.ndarray) and R.ndim>0 and S.ndim>0 and R.shape[1]==S.shape[1]:
                # don't block benign cases here; keep original behavior
                pass
        except Exception:
            pass
    return _compute_pair_metrics_orig(R, S, name, outdir)

# replace the global symbol so all later calls go through guard
compute_pair_metrics = _guarded_compute_pair_metrics

# Métricas de distribución y series temporales
def pca_features(X, n_components=50):
    n_samples, n_times = X.shape
    n_components = min(n_components, n_samples, n_times)
    if n_components <= 0:
        return X
    pca = PCA(n_components=n_components)
    return pca.fit_transform(X)

def discriminative_score(real, synth, n_pca=50, n_splits=5, random_state=SEED):
    X = np.vstack([real, synth])
    y = np.hstack([np.zeros(len(real)), np.ones(len(synth))])
    Xf = pca_features(X, n_components=n_pca)
    try:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        aucs = []
        clf = LogisticRegression(max_iter=2000)
        for train_idx, test_idx in cv.split(Xf, y):
            if len(np.unique(y[test_idx]))<2:
                continue
            clf.fit(Xf[train_idx], y[train_idx])
            probs = clf.predict_proba(Xf[test_idx])[:,1]
            aucs.append(roc_auc_score(y[test_idx], probs))
        return float(np.mean(aucs)) if len(aucs)>0 else float(0.5)
    except Exception:
        return float(0.5)

def predictive_score(real, synth, n_pca=50, n_splits=5, random_state=SEED):
    real = np.asarray(real)
    n_real = len(real)
    if n_real < 2:
        return float(0.5)
    y_real = np.zeros(n_real, dtype=int)
    cv = StratifiedKFold(n_splits=min(n_splits, n_real), shuffle=True, random_state=random_state)
    fold_scores = []
    for train_idx, test_idx in cv.split(real, y_real):
        real_train = real[train_idx]
        real_test = real[test_idx]
        X_train = np.vstack([synth, real_train])
        y_train = np.hstack([np.ones(len(synth)), np.zeros(len(real_train))])
        try:
            n_comp = min(n_pca, X_train.shape[0], X_train.shape[1])
            if n_comp <= 0:
                Xtr_f = X_train
                Xte_f = real_test
            else:
                pca_local = PCA(n_components=n_comp)
                Xtr_f = pca_local.fit_transform(X_train)
                Xte_f = pca_local.transform(real_test)
            clf = LogisticRegression(max_iter=2000)
            clf.fit(Xtr_f, y_train)
            probs = clf.predict_proba(Xte_f)[:,1]
            y_test = np.zeros(len(real_test), dtype=int)
            try:
                s = roc_auc_score(y_test, probs)
                if np.isnan(s):
                    preds = (probs >= 0.5).astype(int)
                    s = accuracy_score(y_test, preds)
            except Exception:
                preds = (probs >= 0.5).astype(int)
                s = accuracy_score(y_test, preds)
            fold_scores.append(float(s))
        except Exception:
            continue
    if len(fold_scores) == 0:
        return float(0.5)
    return float(np.mean(fold_scores))

def marginal_distribution_diff(real, synth):
    return float(wasserstein_distance(real.ravel(), synth.ravel()))

def autocorr(x, max_lag=None):
    x = x - np.mean(x)
    N = len(x)
    corr = correlate(x, x, mode='full')
    corr = corr[N-1:]
    denom = (np.var(x) * np.arange(N, 0, -1))
    denom[denom==0]=1.0
    corr = corr / denom
    if max_lag is None:
        return corr
    return corr[:max_lag]

def autocorr_difference(real, synth, max_lag=300):
    real_ac = np.array([autocorr(x, max_lag=max_lag) for x in real])
    synth_ac = np.array([autocorr(x, max_lag=max_lag) for x in synth])
    mean_real = np.mean(real_ac, axis=0)
    mean_synth = np.mean(synth_ac, axis=0)
    return float(np.mean(np.abs(mean_real - mean_synth)))

def skewness_difference(real, synth):
    r = skew(real, axis=1)
    s = skew(synth, axis=1)
    return float(np.abs(np.mean(r) - np.mean(s)))

def kurtosis_difference(real, synth):
    r = kurtosis(real, axis=1)
    s = kurtosis(synth, axis=1)
    return float(np.abs(np.mean(r) - np.mean(s)))

# C-FID, DTW y embeddings

def compute_fid_from_feats(X, Y):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    mu_x = X.mean(axis=0); mu_y = Y.mean(axis=0)
    cov_x = np.cov(X, rowvar=False); cov_y = np.cov(Y, rowvar=False)
    diff = mu_x - mu_y
    try:
        covmean = sqrtm(cov_x.dot(cov_y))
    except Exception:
        eps = 1e-6
        covmean = sqrtm((cov_x + np.eye(cov_x.shape[0])*eps).dot(cov_y + np.eye(cov_y.shape[0])*eps))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(cov_x + cov_y - 2*covmean)
    return float(np.real(fid))

def extract_context_features_via_psd(A, fs=FS, nperseg=1024, n_comp=64):
    psds = []
    for s in A[:min(len(A), MAX_SAMPLE)]:
        f,p = welch(s, fs=fs, nperseg=nperseg)
        idx = f <= PSD_MAX_FREQ
        psds.append(p[idx])
    psds = np.array(psds)
    if psds.shape[0] == 0:
        return np.zeros((0, min(n_comp, psds.shape[1])))
    feats = np.log10(psds + 1e-12)
    n_comp = min(n_comp, feats.shape[1], feats.shape[0])
    if n_comp < feats.shape[1]:
        pca = SKPCA(n_components=n_comp)
        feats = pca.fit_transform(feats)
    return feats

def contextual_fid(R, S):
    try:
        X = extract_context_features_via_psd(R)
        Y = extract_context_features_via_psd(S)
        if X.shape[0] < 2 or Y.shape[0] < 2:
            return float('nan')
        return compute_fid_from_feats(X, Y)
    except Exception as e:
        print('C-FID error:', e)
        return float('nan')


def dtw_distance(a, b, max_len=300):
    a = np.asarray(a).ravel(); b = np.asarray(b).ravel()
    ka = max(1, math.ceil(len(a)/max_len)); kb = max(1, math.ceil(len(b)/max_len))
    a_ds = a[::ka]; b_ds = b[::kb]
    na = len(a_ds); nb = len(b_ds)
    D = np.full((na+1, nb+1), np.inf); D[0,0] = 0.0
    for i in range(1, na+1):
        for j in range(1, nb+1):
            cost = abs(float(a_ds[i-1]) - float(b_ds[j-1]))
            D[i,j] = cost + min(D[i-1,j], D[i,j-1], D[i-1,j-1])
    return float(D[na, nb] / max(1.0, float(na + nb)))

def avg_dtw_between_sets(R, S, n_pairs=100):
    n = min(len(R), len(S))
    if n == 0:
        return float('nan')
    idxs = np.linspace(0, n-1, min(n, n_pairs)).astype(int)
    vals = []
    for i in idxs:
        try:
            vals.append(dtw_distance(R[i], S[i]))
        except Exception:
            continue
    return float(np.mean(vals)) if len(vals)>0 else float('nan')

# Embedding compacto con fallback a UMAP
def embed_and_silhouette(R, S, outdir, title, n_samples=500):
    os.makedirs(outdir, exist_ok=True)
    n = min(n_samples, len(R), len(S))
    if n < 2:
        return {'tSNE_silhouette': float('nan'), 'tSNE_path': None}
    X = np.vstack([R[:n], S[:n]])
    y = np.hstack([np.zeros(n), np.ones(n)])
    pca = SKPCA(n_components=min(50, X.shape[1]))
    Xp = pca.fit_transform(X)
    # try TSNE then UMAP
    try:
        try:
            ts = TSNE(n_components=2, perplexity=min(30, max(5, int(0.1*n))), max_iter=1000, random_state=SEED, init='pca')
        except TypeError:
            ts = TSNE(n_components=2, perplexity=min(30, max(5, int(0.1*n))), n_iter=1000, random_state=SEED, init='pca')
        emb = ts.fit_transform(Xp)
        sil = float('nan')
        try:
            if len(np.unique(y))>1 and emb.shape[0] > 2:
                sil = float(silhouette_score(emb, y))
        except Exception:
            sil = float('nan')
        p = os.path.join(outdir, 'tsne_scatter.png')
        plt.figure(figsize=(6,5))
        plt.scatter(emb[y==0,0], emb[y==0,1], s=8, alpha=0.6, label='Real', c='C0')
        plt.scatter(emb[y==1,0], emb[y==1,1], s=8, alpha=0.6, label='Synth', c='C1')
        plt.legend(); plt.title(title + ' — t-SNE')
        plt.tight_layout(); plt.savefig(p, dpi=150); plt.close()
        return {'tSNE_silhouette': sil, 'tSNE_path': p}
    except Exception:
        if UMAP_AVAILABLE:
            try:
                reducer = _umap.UMAP(n_components=2, random_state=SEED)
                emb = reducer.fit_transform(Xp)
                sil = float('nan')
                try:
                    if len(np.unique(y))>1 and emb.shape[0] > 2:
                        sil = float(silhouette_score(emb, y))
                except Exception:
                    sil = float('nan')
                p = os.path.join(outdir, 'umap_scatter.png')
                plt.figure(figsize=(6,5))
                plt.scatter(emb[y==0,0], emb[y==0,1], s=8, alpha=0.6, label='Real', c='C0')
                plt.scatter(emb[y==1,0], emb[y==1,1], s=8, alpha=0.6, label='Synth', c='C1')
                plt.legend(); plt.title(title + ' — UMAP')
                plt.tight_layout(); plt.savefig(p, dpi=150); plt.close()
                return {'tSNE_silhouette': sil, 'tSNE_path': p}
            except Exception:
                return {'tSNE_silhouette': float('nan'), 'tSNE_path': None}
        return {'tSNE_silhouette': float('nan'), 'tSNE_path': None}

# Cálculo final de métricas y persistencia
real_af = AFR_p; synth_af = AFS_p; real_nsr = NSRR_p; synth_nsr = NSRS_p

# pair summaries
af_res = safe_compute_pair_metrics(real_af, 'real_af', synth_af, 'synth_af', 'AF_processed_vs_synth', OUT_DIR)
nsr_res = safe_compute_pair_metrics(real_nsr, 'real_nsr', synth_nsr, 'synth_nsr', 'NSR_processed_vs_synth', OUT_DIR)

# DS..KD
af_res.update({'DS':discriminative_score(real_af, synth_af), 'PS':predictive_score(real_af, synth_af), 'MDD':marginal_distribution_diff(real_af, synth_af), 'ACD':autocorr_difference(real_af, synth_af, max_lag=min(500, real_af.shape[1]//2)), 'SD':skewness_difference(real_af, synth_af), 'KD':kurtosis_difference(real_af, synth_af)})
nsr_res.update({'DS':discriminative_score(real_nsr, synth_nsr), 'PS':predictive_score(real_nsr, synth_nsr), 'MDD':marginal_distribution_diff(real_nsr, synth_nsr), 'ACD':autocorr_difference(real_nsr, synth_nsr, max_lag=min(500, real_nsr.shape[1]//2)), 'SD':skewness_difference(real_nsr, synth_nsr), 'KD':kurtosis_difference(real_nsr, synth_nsr)})

# C-FID, embeddings y DTW
af_cfid = contextual_fid(real_af, synth_af)
af_emb = embed_and_silhouette(real_af, synth_af, AF_DIR, 'AF processed vs synth')
af_dtw = avg_dtw_between_sets(real_af, synth_af, n_pairs=100)
af_res.update({'CFID':af_cfid, 'tSNE_silhouette':af_emb.get('tSNE_silhouette'), 'tSNE_path':af_emb.get('tSNE_path'), 'DTW':af_dtw})

nsr_cfid = contextual_fid(real_nsr, synth_nsr)
nsr_emb = embed_and_silhouette(real_nsr, synth_nsr, NSR_DIR, 'NSR processed vs synth')
nsr_dtw = avg_dtw_between_sets(real_nsr, synth_nsr, n_pairs=100)
nsr_res.update({'CFID':nsr_cfid, 'tSNE_silhouette':nsr_emb.get('tSNE_silhouette'), 'tSNE_path':nsr_emb.get('tSNE_path'), 'DTW':nsr_dtw})

# Persistir JSONs y resumen "more"
with open(os.path.join(OUT_DIR,'additional_metrics_AF.json'),'w',encoding='utf-8') as fh: json.dump(af_res, fh, indent=2)
with open(os.path.join(OUT_DIR,'additional_metrics_NSR.json'),'w',encoding='utf-8') as fh: json.dump(nsr_res, fh, indent=2)
df_more = pd.DataFrame([{'class':'AF', **{k:af_res.get(k) for k in ['CFID','tSNE_silhouette','DTW']}},{'class':'NSR', **{k:nsr_res.get(k) for k in ['CFID','tSNE_silhouette','DTW']}}])
df_more.to_csv(os.path.join(OUT_DIR,'additional_metrics_more_summary.csv'), index=False)

# NUEVO: CSV con DS, PS, MDD, ACD, SD, KD (para tabla maestra)
df_add = pd.DataFrame([
    {'class':'AF', **{k:af_res.get(k) for k in ['DS','PS','MDD','ACD','SD','KD']}},
    {'class':'NSR', **{k:nsr_res.get(k) for k in ['DS','PS','MDD','ACD','SD','KD']}}
])
df_add.to_csv(os.path.join(OUT_DIR,'additional_metrics_summary.csv'), index=False)

print(f'Métricas calculadas y guardadas en {OUT_DIR}/')


ValueError: Distribution can't be empty.

In [ ]:
# --- PLOTS: funciones de visualización consolidadas y nuevos gráficos ---
from IPython.display import display, HTML
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import welch

# Generador de mosaicos PSD por muestra
def generate_per_sample_psd_grid(A, outpath, title, ncols=8, nrows=5, fs=FS, nperseg=1024, max_samples=40):
    A = np.asarray(A)
    n = min(len(A), max_samples, ncols*nrows)
    fig, axs = plt.subplots(nrows, ncols, figsize=(ncols*1.8, nrows*1.2))
    axs = axs.flatten()
    for i in range(n):
        f,p = welch(A[i], fs=fs, nperseg=nperseg)
        idx = f <= PSD_MAX_FREQ
        axs[i].plot(f[idx], p[idx], color='C0')
        axs[i].set_xticks([]); axs[i].set_yticks([])
    for j in range(n, len(axs)):
        fig.delaxes(axs[j])
    plt.suptitle(title)
    plt.tight_layout(rect=[0,0,1,0.96])
    outpath = Path(outpath)
    outpath.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(str(outpath), dpi=150)
    plt.close()
    return str(outpath)

# Grid de trazas de ejemplo (tiempo)
def generate_sample_traces_grid(A, outpath, title, ncols=8, nrows=5, max_samples=40):
    A = np.asarray(A)
    n = min(len(A), max_samples, ncols*nrows)
    fig, axs = plt.subplots(nrows, ncols, figsize=(ncols*1.8, nrows*1.2))
    axs = axs.flatten()
    for i in range(n):
        t = np.arange(len(A[i])) / FS
        axs[i].plot(t, A[i], color='C0', linewidth=0.8)
        axs[i].set_xticks([]); axs[i].set_yticks([])
    for j in range(n, len(axs)):
        fig.delaxes(axs[j])
    plt.suptitle(title)
    plt.tight_layout(rect=[0,0,1,0.96])
    outpath = Path(outpath)
    outpath.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(str(outpath), dpi=150)
    plt.close()
    return str(outpath)

# PSD overlay medio (promedio real vs synth)
def plot_psd_overlay(R, S, outdir, fname='psd_overlay.png', fs=FS, nperseg=1024):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    n = min(len(R), len(S), MAX_SAMPLE)
    prs = []
    pss = []
    for r in R[:n]:
        f,p = welch(r, fs=fs, nperseg=nperseg)
        prs.append(p)
    for s in S[:n]:
        f2,p2 = welch(s, fs=fs, nperseg=nperseg)
        pss.append(p2)
    prs = np.array(prs); pss = np.array(pss)
    idx = f <= PSD_MAX_FREQ
    mean_r = np.mean(prs, axis=0)
    mean_s = np.mean(pss, axis=0)
    plt.figure(figsize=(6,4))
    plt.plot(f[idx], mean_r[idx], label='Real', color='C0')
    plt.plot(f[idx], mean_s[idx], label='Synth', color='C1')
    plt.xlabel('Freq (Hz)'); plt.ylabel('PSD'); plt.title('PSD promedio — Real vs Synth')
    plt.legend(); plt.tight_layout()
    p = str(outdir / fname)
    plt.savefig(p, dpi=150); plt.close()
    return p

# Proyección (first PC) densidad / KDE simple
def plot_projection_kde(R, S, outdir, fname='projection_kde.png'):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    rproj = first_pc_proj(R)
    sproj = first_pc_proj(S)
    plt.figure(figsize=(6,4))
    try:
        import seaborn as sns
        sns.kdeplot(rproj, color='C0', label='Real', fill=True)
        sns.kdeplot(sproj, color='C1', label='Synth', fill=True)
    except Exception:
        # fallback a hist
        plt.hist(rproj, bins=50, density=True, alpha=0.5, color='C0', label='Real')
        plt.hist(sproj, bins=50, density=True, alpha=0.5, color='C1', label='Synth')
    plt.title('Proyección (1ª PC) — densidad')
    plt.legend(); plt.tight_layout()
    p = str(outdir / fname)
    plt.savefig(p, dpi=150); plt.close()
    return p

# Scatter PCA 2D
def plot_pca_scatter(R, S, outdir, fname='pca_scatter.png'):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    X = np.vstack([R[:MAX_SAMPLE], S[:MAX_SAMPLE]])
    y = np.hstack([np.zeros(min(len(R),MAX_SAMPLE)), np.ones(min(len(S),MAX_SAMPLE))])
    try:
        from sklearn.decomposition import PCA as _PCA
        pca = _PCA(n_components=2)
        Xp = pca.fit_transform(X)
        plt.figure(figsize=(6,5))
        plt.scatter(Xp[y==0,0], Xp[y==0,1], s=8, alpha=0.6, label='Real', c='C0')
        plt.scatter(Xp[y==1,0], Xp[y==1,1], s=8, alpha=0.6, label='Synth', c='C1')
        plt.legend(); plt.title('PCA 2D — Real vs Synth'); plt.tight_layout()
        p = str(outdir / fname)
        plt.savefig(p, dpi=150); plt.close()
        return p
    except Exception:
        return None

# RMSSD y SDNN por muestra — gráficos tipo violín
def compute_rmssd_sdnn_per_sample(A):
    r = []
    s = []
    for sig in A[:MAX_SAMPLE]:
        sig = np.asarray(sig).ravel()
        if len(sig) < 2:
            r.append(np.nan); s.append(np.nan); continue
        diffs = np.diff(sig)
        rmssd = np.sqrt(np.mean(diffs**2))
        sdnn = float(np.std(sig))
        r.append(rmssd); s.append(sdnn)
    return np.array(r), np.array(s)

def plot_rmssd_sdnn_violin(R, S, outdir):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    rr_R, sd_R = compute_rmssd_sdnn_per_sample(R)
    rr_S, sd_S = compute_rmssd_sdnn_per_sample(S)
    try:
        import seaborn as sns
        import pandas as pd
        df = pd.DataFrame({
            'rmssd': np.concatenate([rr_R, rr_S]),
            'sdnn': np.concatenate([sd_R, sd_S]),
            'class': ['Real']*len(rr_R) + ['Synth']*len(rr_S)
        })
        plt.figure(figsize=(8,4))
        plt.subplot(1,2,1)
        sns.violinplot(x='class', y='rmssd', data=df, palette=['C0','C1'])
        plt.title('RMSSD por clase')
        plt.subplot(1,2,2)
        sns.violinplot(x='class', y='sdnn', data=df, palette=['C0','C1'])
        plt.title('SDNN por clase')
        plt.tight_layout()
        p = str(outdir / 'rmssd_sdnn_violin.png')
        plt.savefig(p, dpi=150); plt.close()
        # also save separate files for compatibility
        return p
    except Exception:
        # fallback: simple boxplots
        plt.figure(figsize=(8,4))
        plt.subplot(1,2,1)
        plt.boxplot([rr_R[~np.isnan(rr_R)], rr_S[~np.isnan(rr_S)]], labels=['Real','Synth'])
        plt.title('RMSSD por clase')
        plt.subplot(1,2,2)
        plt.boxplot([sd_R[~np.isnan(sd_R)], sd_S[~np.isnan(sd_S)]], labels=['Real','Synth'])
        plt.title('SDNN por clase')
        plt.tight_layout()
        p = str(outdir / 'rmssd_sdnn_box.png')
        plt.savefig(p, dpi=150); plt.close()
        return p

# Mostrar comparaciones y generar gráficos adicionales si faltan
def show_comparisons_fixed():
    af_dir = Path(AF_DIR)
    nsr_dir = Path(NSR_DIR)

    # generar plots adicionales si no existen
    # PSD overlay
    if not (af_dir / 'psd_overlay.png').exists():
        try:
            plot_psd_overlay(AFR_p, AFS_p, af_dir)
        except Exception as e:
            print('psd_overlay failed:', e)
    if not (nsr_dir / 'psd_overlay.png').exists():
        try:
            plot_psd_overlay(NSRR_p, NSRS_p, nsr_dir)
        except Exception as e:
            print('psd_overlay failed (NSR):', e)
    # projection kde
    if not (af_dir / 'projection_kde.png').exists():
        try:
            plot_projection_kde(AFR_p, AFS_p, af_dir)
        except Exception as e:
            print('projection_kde failed:', e)
    if not (nsr_dir / 'projection_kde.png').exists():
        try:
            plot_projection_kde(NSRR_p, NSRS_p, nsr_dir)
        except Exception as e:
            print('projection_kde failed (NSR):', e)
    # PCA scatter
    if not (af_dir / 'pca_scatter.png').exists():
        try:
            plot_pca_scatter(AFR_p, AFS_p, af_dir)
        except Exception as e:
            print('pca_scatter failed:', e)
    if not (nsr_dir / 'pca_scatter.png').exists():
        try:
            plot_pca_scatter(NSRR_p, NSRS_p, nsr_dir)
        except Exception as e:
            print('pca_scatter failed (NSR):', e)
    # RMSSD/SDNN violín
    if not (af_dir / 'rmssd_violin.png').exists() and not (af_dir / 'rmssd_sdnn_violin.png').exists():
        try:
            plot_rmssd_sdnn_violin(AFR_p, AFS_p, af_dir)
        except Exception as e:
            print('rmssd_violin failed:', e)
    if not (nsr_dir / 'rmssd_violin.png').exists() and not (nsr_dir / 'rmssd_sdnn_violin.png').exists():
        try:
            plot_rmssd_sdnn_violin(NSRR_p, NSRS_p, nsr_dir)
        except Exception as e:
            print('rmssd_violin failed (NSR):', e)
    # per-sample PSD mosaics
    if not (af_dir / 'AF_real_per_sample_psd.png').exists():
        try:
            generate_per_sample_psd_grid(AFR_p, af_dir / 'AF_real_per_sample_psd.png', 'AF_real — per-sample PSD')
        except Exception as e:
            print('generate AF PSD mosaic failed:', e)
    if not (af_dir / 'AF_synth_per_sample_psd.png').exists():
        try:
            generate_per_sample_psd_grid(AFS_p, af_dir / 'AF_synth_per_sample_psd.png', 'AF_synth — per-sample PSD')
        except Exception as e:
            print('generate AF synth PSD mosaic failed:', e)
    if not (nsr_dir / 'NSR_real_per_sample_psd.png').exists():
        try:
            generate_per_sample_psd_grid(NSRR_p, nsr_dir / 'NSR_real_per_sample_psd.png', 'NSR_real — per-sample PSD')
        except Exception as e:
            print('generate NSR PSD mosaic failed:', e)
    if not (nsr_dir / 'NSR_synth_per_sample_psd.png').exists():
        try:
            generate_per_sample_psd_grid(NSRS_p, nsr_dir / 'NSR_synth_per_sample_psd.png', 'NSR_synth — per-sample PSD')
        except Exception as e:
            print('generate NSR synth PSD mosaic failed:', e)
    # sample traces grid
    if not (af_dir / 'sample_traces_grid.png').exists():
        try:
            generate_sample_traces_grid(AFR_p, af_dir / 'sample_traces_grid.png', 'AF sample traces')
        except Exception as e:
            print('generate AF traces failed:', e)
    if not (nsr_dir / 'sample_traces_grid.png').exists():
        try:
            generate_sample_traces_grid(NSRR_p, nsr_dir / 'sample_traces_grid.png', 'NSR sample traces')
        except Exception as e:
            print('generate NSR traces failed:', e)

    # now display a selection of images side-by-side (keeps previous behavior)
    pairs = [
        (('AF_real_per_sample_psd.png','AF_synth_per_sample_psd.png'),'AF per-sample PSD (Real vs Synth)', af_dir),
        (('NSR_real_per_sample_psd.png','NSR_synth_per_sample_psd.png'),'NSR per-sample PSD (Real vs Synth)', nsr_dir),
        ('psd_overlay.png','PSD overlay', None),
        ('projection_kde.png','Projection KDE', None),
        ('pca_scatter.png','PCA scatter', None),
        ('sample_traces_grid.png','Sample traces', None),
        ('rmssd_sdnn_violin.png','RMSSD/SDNN violin', None)
    ]

    for item in pairs:
        if isinstance(item[0], (list, tuple)):
            (left_name, right_name), title, preferred_dir = item
            if preferred_dir is None:
                left_candidates = [af_dir/left_name, nsr_dir/left_name]
                right_candidates = [af_dir/right_name, nsr_dir/right_name]
            else:
                left_candidates = [preferred_dir/left_name]
                right_candidates = [preferred_dir/right_name]
            left_path = next((p for p in left_candidates if p.exists()), None)
            right_path = next((p for p in right_candidates if p.exists()), None)
            left_html = f"<div style='flex:1;text-align:center'><h4>{title} — Left</h4>" + (f"<img src='{left_path.as_posix()}' style='max-width:100%;height:auto'>" if left_path else "<div style='color:#888'>(no disponible)</div>") + "</div>"
            right_html = f"<div style='flex:1;text-align:center'><h4>{title} — Right</h4>" + (f"<img src='{right_path.as_posix()}' style='max-width:100%;height:auto'>" if right_path else "<div style='color:#888'>(no disponible)</div>") + "</div>"
            display(HTML('<div style="display:flex;gap:12px;align-items:flex-start">' + left_html + right_html + '</div><hr>'))
        else:
            fname, title, _ = item
            left_path = af_dir/fname
            right_path = nsr_dir/fname
            left_exists = left_path.exists()
            right_exists = right_path.exists()
            left_html = f"<div style='flex:1;text-align:center'><h4>{title} — AF</h4>" + (f"<img src='{left_path.as_posix()}' style='max-width:100%;height:auto'>" if left_exists else "<div style='color:#888'>(no disponible)</div>") + "</div>"
            right_html = f"<div style='flex:1;text-align:center'><h4>{title} — NSR</h4>" + (f"<img src='{right_path.as_posix()}' style='max-width:100%;height:auto'>" if right_exists else "<div style='color:#888'>(no disponible)</div>") + "</div>"
            display(HTML('<div style="display:flex;gap:12px;align-items:flex-start">' + left_html + right_html + '</div><hr>'))

print('Funciones de plot ampliadas disponibles: llama a show_comparisons_fixed() para generar y ver más gráficos.')


Funciones de plot ampliadas disponibles: llama a show_comparisons_fixed() para generar y ver más gráficos.


In [ ]:
# Celda: Generar y mostrar violines RMSSD/SDNN inline
# Si los PNG no existen, se regeneran aquí (no cambia otras partes del notebook).
from IPython.display import display, Image, HTML
from pathlib import Path

force_regenerate_violins = False  # pon True para forzar regeneración

# Use the canonical AF_DIR/NSR_DIR created earlier
af_dir = Path(AF_DIR)
nsr_dir = Path(NSR_DIR)
af_dir.mkdir(parents=True, exist_ok=True)
nsr_dir.mkdir(parents=True, exist_ok=True)

files_to_show = [
    ('AF', af_dir / 'rmssd_sdnn_violin.png', lambda: plot_rmssd_sdnn_violin(AFR_p, AFS_p, af_dir)),
    ('AF', af_dir / 'rmssd_violin.png', lambda: plot_rmssd_sdnn_violin(AFR_p, AFS_p, af_dir)),
    ('AF', af_dir / 'sdnn_violin.png', lambda: plot_rmssd_sdnn_violin(AFR_p, AFS_p, af_dir)),
    ('NSR', nsr_dir / 'rmssd_sdnn_violin.png', lambda: plot_rmssd_sdnn_violin(NSRR_p, NSRS_p, nsr_dir)),
    ('NSR', nsr_dir / 'rmssd_violin.png', lambda: plot_rmssd_sdnn_violin(NSRR_p, NSRS_p, nsr_dir)),
    ('NSR', nsr_dir / 'sdnn_violin.png', lambda: plot_rmssd_sdnn_violin(NSRR_p, NSRS_p, nsr_dir)),
]

for cls, p, regen in files_to_show:
    try:
        if force_regenerate_violins or not p.exists():
            regen()
    except Exception as e:
        print(f'No se pudo regenerar {p.name}:', e)


In [ ]:
# Diagnóstico: listar archivos en carpetas de salida y comprobar archivos esperados
import os
from pathlib import Path

def list_out_files():
    dirs = [AF_DIR, NSR_DIR]
    for d in dirs:
        print('==', d, '==')
        if not os.path.exists(d):
            print('  (no existe)')
            continue
        for p in sorted(Path(d).glob('*')):
            print(' ', p.name)
    expected = [
        'AF_real_per_sample_psd.png','AF_synth_per_sample_psd.png',
        'NSR_real_per_sample_psd.png','NSR_synth_per_sample_psd.png'
    ]
    print('Existence check for expected per-sample PSD files:')
    for e in expected:
        found = any(Path(d).joinpath(e).exists() for d in dirs)
        print(f' {e}:', 'FOUND' if found else 'MISSING')

list_out_files()


== compare_out_p09\pretty_AF ==
  AF_real_per_sample_psd.png
  AF_synth_per_sample_psd.png
  pca_scatter.png
  projection_kde.png
  psd_overlay.png
  rmssd_sdnn_violin.png
  sample_traces_grid.png
  tsne_scatter.png
== compare_out_p09\pretty_NSR ==
  NSR_real_per_sample_psd.png
  NSR_synth_per_sample_psd.png
  pca_scatter.png
  projection_kde.png
  psd_overlay.png
  rmssd_sdnn_violin.png
  sample_traces_grid.png
  tsne_scatter.png
Existence check for expected per-sample PSD files:
 AF_real_per_sample_psd.png: FOUND
 AF_synth_per_sample_psd.png: FOUND
 NSR_real_per_sample_psd.png: FOUND
 NSR_synth_per_sample_psd.png: FOUND


In [ ]:
# Llamada sencilla a la función de comparaciones consolidadas.
print('Mostrando comparaciones (versión consolidada).')
show_comparisons_fixed()


Mostrando comparaciones (versión consolidada).


In [ ]:
# Agregar y mostrar una tabla maestra con todas las métricas calculadas (redondeadas a 4 decimales)
import os, json
import pandas as pd
from IPython.display import display, HTML

out_dir = OUT_DIR
# Leer CSVs resumen si están disponibles
try:
    df_add = pd.read_csv(os.path.join(out_dir, 'additional_metrics_summary.csv'))  # DS..KD
except Exception:
    df_add = pd.DataFrame()
try:
    df_more = pd.read_csv(os.path.join(out_dir, 'additional_metrics_more_summary.csv'))  # CFID, tSNE, DTW
except Exception:
    df_more = pd.DataFrame()

# helper para leer archivos de resumen por par (formato metric,value)
def read_pair_summary(p):
    if not os.path.exists(p):
        return {}
    try:
        tmp = pd.read_csv(p)
        if 'metric' in tmp.columns and 'value' in tmp.columns:
            return tmp.set_index('metric')['value'].to_dict()
        # fallback a tabla ancha
        return tmp.to_dict(orient='records')[0] if len(tmp)>0 else {}
    except Exception:
        return {}

# cargar pares
af_sum = read_pair_summary(os.path.join(out_dir, 'AF_processed_vs_synth_summary.csv'))
nsr_sum = read_pair_summary(os.path.join(out_dir, 'NSR_processed_vs_synth_summary.csv'))

# fallback: si df_add está vacío, lee desde los JSONs completos
def read_json_metrics(path):
    try:
        with open(path, 'r', encoding='utf-8') as fh:
            return json.load(fh)
    except Exception:
        return {}

af_json = read_json_metrics(os.path.join(out_dir, 'additional_metrics_AF.json'))
nsr_json = read_json_metrics(os.path.join(out_dir, 'additional_metrics_NSR.json'))

classes = ['AF', 'NSR']
rows = []
for cls in classes:
    row = {'class': cls}
    # métricas principales DS..KD
    if not df_add.empty and 'class' in df_add.columns:
        r = df_add[df_add['class'] == cls]
        if len(r) > 0:
            for c in ['DS','PS','MDD','ACD','SD','KD']:
                row[c] = r.iloc[0].get(c, float('nan'))
    else:
        src = af_json if cls=='AF' else nsr_json
        for c in ['DS','PS','MDD','ACD','SD','KD']:
            row[c] = src.get(c, float('nan'))

    # CFID / tSNE / DTW
    if not df_more.empty and 'class' in df_more.columns:
        r = df_more[df_more['class'] == cls]
        if len(r) > 0:
            for c in ['CFID','tSNE_silhouette','DTW']:
                row[c] = r.iloc[0].get(c, float('nan'))
    else:
        src = af_json if cls=='AF' else nsr_json
        for c in ['CFID','tSNE_silhouette','DTW']:
            row[c] = src.get(c, float('nan'))

    # resúmenes por par: añadir también ks_stat, ks_p, wasserstein, mean_diff_proj, además de los existentes
    src = af_sum if cls == 'AF' else nsr_sum
    for k in ['mmd2','mmd_sigma','energy','psd_l2','discriminator_auc','ks_stat','ks_p','wasserstein','mean_diff_proj']:
        row[k] = src.get(k, float('nan'))
    rows.append(row)

metrics_df = pd.DataFrame(rows)
# orden preferido de columnas (incluye todas)
cols = ['class','DS','PS','MDD','ACD','SD','KD','CFID','tSNE_silhouette','DTW',
        'mmd2','mmd_sigma','energy','psd_l2','discriminator_auc','ks_stat','ks_p','wasserstein','mean_diff_proj']
metrics_df = metrics_df[[c for c in cols if c in metrics_df.columns]]

# Redondear columnas numéricas a 4 decimales
if 'class' in metrics_df.columns:
    num_cols = metrics_df.columns.drop('class')
else:
    num_cols = metrics_df.columns
metrics_df[num_cols] = metrics_df[num_cols].apply(pd.to_numeric, errors='coerce').round(4)

# Guardar CSV y HTML
os.makedirs(out_dir, exist_ok=True)
csv_path = os.path.join(out_dir, 'metrics_master_table.csv')
html_path = os.path.join(out_dir, 'metrics_master_table.html')
metrics_df.to_csv(csv_path, index=False)

# Render HTML bonito
try:
    styled = metrics_df.style.format(na_rep='-', formatter="{:.4f}").set_table_attributes('style="width:100%;border-collapse:collapse"').render()
    with open(html_path, 'w', encoding='utf-8') as fh:
        fh.write('<meta charset="utf-8">\n')
        fh.write('<style>table{font-family:Arial,Helvetica,sans-serif;border:1px solid #ccc;} td, th{padding:6px;text-align:right;border:1px solid #eee;}</style>\n')
        fh.write('<h2>Metrics Master Table</h2>\n')
        fh.write(styled)
except Exception:
    with open(html_path, 'w', encoding='utf-8') as fh:
        fh.write('<meta charset="utf-8">\n')
        fh.write(metrics_df.to_html(index=False))

# Mostrar inline
display(HTML('<h2>Metrics Master Table</h2>'))
display(HTML(metrics_df.to_html(index=False, float_format="{:.4f}".format)))
print(f'Saved CSV: {csv_path}\nSaved HTML: {html_path}')


NameError: name 'OUT_DIR' is not defined

In [ ]:
# Mostrar tabla P09/P10 vs Article (si existe)
import os
import pandas as pd
from IPython.display import display, HTML
csvp = os.path.join('compare_out_generated', 'p09_p10_vs_article_metrics.csv')
htmlp = os.path.join('compare_out_generated', 'p09_p10_vs_article_metrics.html')
if os.path.exists(csvp):
    df = pd.read_csv(csvp)
    display(HTML('<h2>P09 / P10 vs Article — Summary</h2>'))
    display(HTML(df.to_html(index=False, float_format="{:.4f}".format)))
    print('Saved CSV:', csvp)
else:
    print('No P09/P10 vs Article CSV found at', csvp)
if os.path.exists(htmlp):
    print('Also saved HTML at', htmlp)

No P09/P10 vs Article CSV found at compare_out_generated\p09_p10_vs_article_metrics.csv


In [ ]:
# Mostrar ambas tablas generadas (P09 processed vs Article y P09/P10 vs Article)
import os, pandas as pd
from IPython.display import display, HTML
def show_csv(name, csvpath, htmlpath=None, title=None):
    if os.path.exists(csvpath):
        df = pd.read_csv(csvpath)
        display(HTML(f'<h2>{title or name}</h2>'))
        display(HTML(df.to_html(index=False, float_format="{:.4f}".format)))
        print('Saved CSV:', csvpath)
        if htmlpath and os.path.exists(htmlpath):
            print('Also saved HTML at', htmlpath)
    else:
        print('Missing', csvpath)

show_csv('P09 processed vs Article', os.path.join('compare_out_generated','p09_processed_vs_article_metrics.csv'), os.path.join('compare_out_generated','p09_processed_vs_article_metrics.html'), 'P09 Processed vs Article — Summary')
show_csv('P09/P10 vs Article', os.path.join('compare_out_generated','p09_p10_vs_article_metrics.csv'), os.path.join('compare_out_generated','p09_p10_vs_article_metrics.html'), 'P09 / P10 vs Article — Summary')

Missing compare_out_generated\p09_processed_vs_article_metrics.csv
Missing compare_out_generated\p09_p10_vs_article_metrics.csv
